# Token Pipeline Inspector

Set a `SEQUENCE`, run it, and watch the effect on `State` at each step: what
each token **takes** (its resolved inputs) and its **impact** (new/changed
features, board signals, scale/transform changes, prediction + residual).

All machinery lives in `pipeline_inspect.py` — this notebook is just config and
calls.


## Setup

In [ ]:
from pathlib import Path
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k):
        return x
warnings.filterwarnings('default')
%matplotlib inline

HERE = Path.cwd().resolve()
for root in [HERE, HERE.parent, HERE.parent.parent]:
    if (root / "graph_Time_series" / "state.py").exists():
        PACKAGE_ROOT = root; break
    if (root / "graph_Time_series" / "graph_Time_series" / "state.py").exists():
        PACKAGE_ROOT = root / "graph_Time_series"; break
else:
    raise FileNotFoundError("Could not locate the graph_Time_series package checkout.")
sys.path.insert(0, str(PACKAGE_ROOT))
sys.path.insert(0, str(PACKAGE_ROOT / "examples"))
sys.path.insert(0, str(HERE))

from pipeline_inspect import (
    inspect_pipeline, build_grammar, token_table,
    summarize_state, signal_board_df, feature_bundle_df, artifact_df,
    plot_raw_data, plot_array_artifact, plot_prediction_state, score_prediction,
)

grammar = build_grammar()
print(grammar)


## Data

Produce `H` (history) and `F` (future) arrays of shape `(n_samples, length)`.

In [ ]:
HISTORY_LENGTH = 3_000
HORIZON = 700
SEASONAL_PERIOD = 48
SEED = 7

# Data source options: "pretrain_extracted_chunks", "pretrain_sliding_from_extracted", "manual_series", "synthetic".
DATA_SOURCE = "pretrain_extracted_chunks"

# Local pretrain shards created by nested loops/pretrain_chunk_builder.ipynb.
PRETRAIN_RUN_DIR = (
    PACKAGE_ROOT.parent
    / "nested loops"
    / "pretrain_extracted_shards"
    / "GiftEvalPretrain__Energy-Transport__20260607_154145"
)
PRETRAIN_TAG_CONTAINS = "Energy__"  # use "australian_electricity_demand" for one subset, or "" for all shards
PRETRAIN_MAX_RECORDS = None          # None = all records matching PRETRAIN_TAG_CONTAINS
PRETRAIN_SERIES_MODE = "history_plus_future"  # only used by pretrain_sliding_from_extracted

# Sliding-window construction, only used by manual_series or pretrain_sliding_from_extracted.
SLIDING_STRIDE = HORIZON
MAX_WINDOWS_TOTAL = None
MAX_WINDOWS_PER_SOURCE_SERIES = None
DROP_SHORT_SERIES = True

# Manual series mode. Set DATA_SOURCE = "manual_series" and assign SERIES.
# Example: SERIES = np.asarray(my_dataframe["value"].values, dtype=np.float32)
SERIES = None

def clean_1d_series(values):
    values = np.asarray(values, dtype=np.float32).reshape(-1).copy()
    if values.size == 0:
        return values
    finite = np.isfinite(values)
    if np.all(finite):
        return values
    if not np.any(finite):
        return np.zeros_like(values, dtype=np.float32)
    idx = np.arange(values.size)
    values[~finite] = np.interp(idx[~finite], idx[finite], values[finite]).astype(np.float32)
    return values

def sliding_windows_from_series(series, *, history_len, horizon, stride, max_windows=None, meta=None):
    series = clean_1d_series(series)
    needed = history_len + horizon
    if series.size < needed:
        if DROP_SHORT_SERIES:
            return []
        raise ValueError(f"Series has {series.size} points, need at least {needed}.")
    starts = range(0, series.size - needed + 1, max(int(stride), 1))
    windows = []
    for local_idx, start in enumerate(starts):
        if max_windows is not None and local_idx >= max_windows:
            break
        end_h = start + history_len
        end_f = end_h + horizon
        windows.append({
            "history": series[start:end_h].astype(np.float32, copy=True),
            "future": series[end_h:end_f].astype(np.float32, copy=True),
            "window_start": int(start),
            "window_end": int(end_f),
            **(meta or {}),
        })
    return windows

def load_pretrain_records(run_dir, tag_contains="", max_records=None):
    run_dir = Path(run_dir)
    manifest_path = run_dir / "shard_manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing shard manifest: {manifest_path}")
    manifest = pd.read_csv(manifest_path)
    if tag_contains:
        mask = manifest["tag"].astype(str).str.contains(tag_contains, case=False, regex=False)
        manifest = manifest[mask].copy()
    if manifest.empty:
        raise ValueError(f"No pretrain shards matched tag_contains={tag_contains!r} in {manifest_path}")

    records = []
    selected_rows = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="loading pretrain shards"):
        shard_path = Path(row["path"])
        if not shard_path.exists():
            shard_path = run_dir / "extracted_chunks" / shard_path.name
        with shard_path.open("rb") as f:
            shard_records = pickle.load(f)
        for rec in shard_records:
            records.append(rec)
            selected_rows.append(row.to_dict())
            if max_records is not None and len(records) >= max_records:
                return records, pd.DataFrame(selected_rows)
    return records, pd.DataFrame(selected_rows)

def pretrain_record_to_series(record, mode="history_plus_future"):
    history = clean_1d_series(record.get("history", []))
    future = clean_1d_series(record.get("future", []))
    if mode == "history_only":
        return history
    if mode == "future_only":
        return future
    if mode == "history_plus_future":
        return np.concatenate([history, future]).astype(np.float32)
    raise ValueError("PRETRAIN_SERIES_MODE must be history_only, future_only, or history_plus_future")

def build_dataset_from_pretrain_extracted_chunks():
    records, selected_manifest = load_pretrain_records(
        PRETRAIN_RUN_DIR,
        tag_contains=PRETRAIN_TAG_CONTAINS,
        max_records=PRETRAIN_MAX_RECORDS,
    )
    rows, histories, futures = [], [], []
    for record_idx, rec in enumerate(tqdm(records, desc="stacking extracted chunks")):
        history = clean_1d_series(rec.get("history", []))
        future = clean_1d_series(rec.get("future", []))
        if HISTORY_LENGTH is not None and history.size != HISTORY_LENGTH:
            continue
        if HORIZON is not None and future.size != HORIZON:
            continue
        histories.append(history.astype(np.float32, copy=True))
        futures.append(future.astype(np.float32, copy=True))
        rows.append({
            "source": "pretrain_extracted_chunks",
            "record_idx": record_idx,
            "sample_idx": rec.get("sample_idx"),
            "source_domain": rec.get("source_domain"),
            "source_subset": rec.get("source_subset"),
            "source_row_idx": rec.get("source_row_idx"),
            "item_id": rec.get("item_id"),
            "freq": rec.get("freq"),
            "chunk_start": rec.get("chunk_start"),
            "history_len": int(history.size),
            "future_len": int(future.size),
            "source_length": rec.get("source_length"),
            "history_missing_fraction": rec.get("history_missing_fraction"),
            "future_missing_fraction": rec.get("future_missing_fraction"),
        })
    if not histories:
        raise ValueError("No extracted chunks matched the selected length/filter settings.")
    return (
        np.stack(histories).astype(np.float32),
        np.stack(futures).astype(np.float32),
        pd.DataFrame(rows),
        selected_manifest,
    )

def build_windows_from_pretrain_shards():
    records, selected_manifest = load_pretrain_records(
        PRETRAIN_RUN_DIR,
        tag_contains=PRETRAIN_TAG_CONTAINS,
        max_records=PRETRAIN_MAX_RECORDS,
    )
    all_windows = []
    iterator = tqdm(records, desc="cutting sliding windows")
    for record_idx, rec in enumerate(iterator):
        series = pretrain_record_to_series(rec, PRETRAIN_SERIES_MODE)
        meta = {
            "source": "pretrain_shard",
            "record_idx": record_idx,
            "sample_idx": rec.get("sample_idx"),
            "source_domain": rec.get("source_domain"),
            "source_subset": rec.get("source_subset"),
            "source_row_idx": rec.get("source_row_idx"),
            "item_id": rec.get("item_id"),
            "freq": rec.get("freq"),
            "series_len": int(series.size),
        }
        windows = sliding_windows_from_series(
            series,
            history_len=HISTORY_LENGTH,
            horizon=HORIZON,
            stride=SLIDING_STRIDE,
            max_windows=MAX_WINDOWS_PER_SOURCE_SERIES,
            meta=meta,
        )
        all_windows.extend(windows)
        iterator.set_postfix(windows=len(all_windows))
        if MAX_WINDOWS_TOTAL is not None and len(all_windows) >= MAX_WINDOWS_TOTAL:
            all_windows = all_windows[:MAX_WINDOWS_TOTAL]
            break
    if not all_windows:
        raise ValueError("No sliding windows were created from the selected pretrain records.")
    H_arr = np.stack([w["history"] for w in all_windows]).astype(np.float32)
    F_arr = np.stack([w["future"] for w in all_windows]).astype(np.float32)
    meta_df = pd.DataFrame([{k: v for k, v in w.items() if k not in {"history", "future"}} for w in all_windows])
    return H_arr, F_arr, meta_df, selected_manifest

def make_synthetic_data(n_samples=8, hist_len=HISTORY_LENGTH, horizon=HORIZON, period=SEASONAL_PERIOD, seed=SEED):
    rng = np.random.default_rng(seed)
    t_hist = np.arange(hist_len)
    t_future = np.arange(hist_len, hist_len + horizon)
    histories, futures = [], []
    for i in range(n_samples):
        level = 10.0 + 0.6 * i
        amp = 2.2 + 0.2 * np.sin(i)
        trend = 0.01 * (i + 1)
        hist = (
            level
            + trend * t_hist
            + amp * np.sin(2 * np.pi * t_hist / period)
            + 0.6 * np.cos(2 * np.pi * t_hist / 7)
            + rng.normal(0, 0.25, hist_len)
        )
        fut = (
            level
            + trend * t_future
            + amp * np.sin(2 * np.pi * t_future / period)
            + 0.6 * np.cos(2 * np.pi * t_future / 7)
        )
        histories.append(hist)
        futures.append(fut)
    return np.asarray(histories, np.float32), np.asarray(futures, np.float32)

if DATA_SOURCE == "pretrain_extracted_chunks":
    H, F, WINDOW_META, PRETRAIN_SELECTED_MANIFEST = build_dataset_from_pretrain_extracted_chunks()
elif DATA_SOURCE == "pretrain_sliding_from_extracted":
    H, F, WINDOW_META, PRETRAIN_SELECTED_MANIFEST = build_windows_from_pretrain_shards()
elif DATA_SOURCE == "manual_series":
    if SERIES is None:
        raise ValueError("Set SERIES when DATA_SOURCE='manual_series'.")
    windows = sliding_windows_from_series(
        SERIES,
        history_len=HISTORY_LENGTH,
        horizon=HORIZON,
        stride=SLIDING_STRIDE,
        max_windows=MAX_WINDOWS_TOTAL,
        meta={"source": "manual_series"},
    )
    H = np.stack([w["history"] for w in windows]).astype(np.float32)
    F = np.stack([w["future"] for w in windows]).astype(np.float32)
    WINDOW_META = pd.DataFrame([{k: v for k, v in w.items() if k not in {"history", "future"}} for w in windows])
    PRETRAIN_SELECTED_MANIFEST = pd.DataFrame()
elif DATA_SOURCE == "synthetic":
    H, F = make_synthetic_data(n_samples=min(MAX_WINDOWS_TOTAL or 8, 64))
    WINDOW_META = pd.DataFrame({"source": ["synthetic"] * H.shape[0], "record_idx": np.arange(H.shape[0])})
    PRETRAIN_SELECTED_MANIFEST = pd.DataFrame()
else:
    raise ValueError("DATA_SOURCE must be pretrain_extracted_chunks, pretrain_sliding_from_extracted, manual_series, or synthetic")

print("DATA_SOURCE", DATA_SOURCE)
print("H", H.shape, "F", F.shape)
if not PRETRAIN_SELECTED_MANIFEST.empty:
    display(PRETRAIN_SELECTED_MANIFEST.drop_duplicates("path").head(10))
display(WINDOW_META.head(12))
pd.DataFrame({"history_first_sample": H[0], "future_first_sample": np.r_[F[0], np.full(max(H.shape[1] - F.shape[1], 0), np.nan)[:H.shape[1]]] if F.shape[1] <= H.shape[1] else H[0]}).head()

In [ ]:
plot_raw_data(H, F, sample_idx=0)

## Choose a pipeline
[
  [1, "ZNormalization", "kernel_rbf", "# works OK"],
  [2, "MeanAbsScaling", "rf_tabular", "# works OK"],
  [3, "ZNormalization", "FourierFeatures", "rf_tabular", "# Fourier + RF"],
  [4, "MeanAbsScaling", "FourierFeatures", "lightgbm_tabular", "# LightGBM tabular"],
  [5, "ZNormalization", "PeriodDetect", "SeasonalFeatures", "step_regression", "# period features + step regression"],
  [6, "ZNormalization", "PeriodDetect", "SeasonalFeatures", "rf_tabular", "# period features + RF"],
  [7, "ZNormalization", "PeriodDetect", "SeasonalFeatures", "kernel_rbf", "# period features + RBF"],
  [8, "LinearFill", "PeriodDetect", "SeasonalFold", "level_shape_ridge", "# clean + recursive seasonal fold + level/shape ridge"],
  [9, "LinearFill", "PeriodDetect", "PeriodPhaseOneHot", "SeasonalFold", "LevelBoxCoxCenter", "FlairRidgeLevel", "# verbose inspection path"],
  [10, "LinearFill", "PeriodDetect", "SeasonalFold", "gb_level_forecast", "# gradient-boosted level model swap"],
  [11, "ZNormalization", "parrot", "kernel_rbf", "# parrot first, kernel fits residual"],
  [12, "ZNormalization", "kernel_rbf", "parrot", "# parrot after another model"],
  [13, "MeanAbsScaling", "FourierFeatures", "parrot", "# parrot reads scaled history, Fourier is just upstream context"],
  [14, "LinearFill", "PeriodDetectSpectral", "SeasonalFold", "level_shape_ridge", "# spectral detector variant"]
]


In [ ]:
# --- residual-stack test (edit SEQUENCE to try others) ---
SEQUENCE = ["ZNormalization", "parrot"]

# other sequences to try:
# SEQUENCE = ["ZNormalization", "kernel_rbf", "parrot"]
# SEQUENCE = ["MeanAbsScaling", "FourierFeatures", "parrot"]
# SEQUENCE = ["LinearFill", "PeriodDetectSpectral", "SeasonalFold", "level_shape_ridge"]
# SEQUENCE = ["LinearFill", "PeriodDetect", "SeasonalFold", "LevelBoxCoxCenter", "FlairRidgeLevel"]

SAMPLE_IDX      = 0
SEASONAL_PERIOD = 48
PLOT_EACH_STEP  = True
SHOW_SIGNAL_BOARD = False
SHOW_FEATURE_BUNDLE = False


## Run the pipeline

Each token prints what it **takes** and its **impact** on `State`.

In [ ]:
state, states = inspect_pipeline(
    SEQUENCE, H, F,
    grammar=grammar,
    seasonal_period=SEASONAL_PERIOD,
    sample_idx=1,
    plot_each_step=PLOT_EACH_STEP,
    show_signal_board=SHOW_SIGNAL_BOARD,
    show_feature_bundle=SHOW_FEATURE_BUNDLE,
)


## Final forecast

In [ ]:
plot_prediction_state(state, sample_idx= 3, title="Final decoded prediction")
score_prediction(state)

## Deep dive (optional)

`states` holds a copy after every token: `states[1]` is after the first token, `states[-1]` is final.

In [ ]:
label, st = states[-1]
print("Inspecting:", label)
summarize_state(st)
display(signal_board_df(st))
display(feature_bundle_df(st))
display(artifact_df(st))


## Token reference

In [ ]:
token_table(seasonal_period=SEASONAL_PERIOD)